# Salt River Model Selection and Hyperparameter Tuning

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import itertools
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from Baselines.baselines import SeasonalAverage
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import os
import joblib
from sklearn.inspection import PartialDependenceDisplay

import warnings
warnings.filterwarnings("ignore")  

In [2]:
df = pd.read_csv("../Data/FormattedData/Combined_Monthly_Data.csv")
season_length = 12
test_size = int(8.5 * season_length)  # Set aside the last 8.5 years as our ultimate testing set.
df_train = df[:-test_size]
df_test = df[-test_size:]

OUTPUT_DIR = "../Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Shared helper functions used for both Salt upstream and downstream below.

In [3]:
def recursive_cv_rmse(model_builder, features, target, lag_prefix, data, tscv,
                       _cache={}):
    """3-fold recursive time-series CV RMSE. Within each held-out fold, any
    lag feature that would require a value from *inside* that fold is
    replaced with our own prior prediction rather than the true historical
    value - this simulates genuine forward forecasting, where you don't
    actually know next month's answer yet.

    Uses raw numpy arrays (not per-row pandas indexing) for speed - this
    is called thousands of times across lag selection + grid search, and
    row-by-row pandas .loc/.iloc access adds up fast at that scale."""
    key = (id(data), tuple(features))
    if key not in _cache:
        _cache[key] = data[features].to_numpy(dtype=float)
    X_all = _cache[key]
    y_all = data[target].to_numpy(dtype=float)
    lag_positions = {i: features.index(f"{lag_prefix}_lag_{i}") for i in range(1, 13)
                     if f"{lag_prefix}_lag_{i}" in features}

    fold_rmses = []
    for train_idx, test_idx in tscv.split(data):
        model = model_builder()
        model.fit(X_all[train_idx], y_all[train_idx])

        X_ho = X_all[test_idx].copy()
        y_ho = y_all[test_idx]
        n = len(test_idx)
        preds = np.empty(n)
        for month in range(n):
            row = X_ho[month]
            for lag_idx, col_pos in lag_positions.items():
                if month >= lag_idx:
                    row[col_pos] = preds[month - lag_idx]
            preds[month] = model.predict(row.reshape(1, -1))[0]
        fold_rmses.append(root_mean_squared_error(y_ho, preds))
    return np.mean(fold_rmses)


def select_lags(model_builder, core_features, lag_pool, target, lag_prefix, data, tscv, n_lags=2):
    """Greedy forward selection: repeatedly add whichever remaining lag
    reduces recursive CV RMSE the most."""
    current_features = core_features.copy()
    remaining = lag_pool.copy()
    chosen = []
    best_score = None
    for step in range(n_lags):
        step_best_score, step_best_lag = float("inf"), None
        for lag in remaining:
            score = recursive_cv_rmse(model_builder, current_features + [lag], target, lag_prefix, data, tscv)
            if score < step_best_score:
                step_best_score, step_best_lag = score, lag
        current_features.append(step_best_lag)
        chosen.append(step_best_lag)
        remaining.remove(step_best_lag)
        best_score = step_best_score
    return chosen, best_score


def grid_search(model_cls, static_params, grid, core_features, lags, target, lag_prefix, data, tscv):
    features = core_features + lags
    keys, values = list(grid.keys()), list(grid.values())
    best_score, best_params = float("inf"), None
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        builder = lambda p=params: model_cls(**{**static_params, **p})
        score = recursive_cv_rmse(builder, features, target, lag_prefix, data, tscv)
        if score < best_score:
            best_score, best_params = score, params
    return best_params, best_score


def evaluate_on_test_set(model, features, target, lag_prefix, df_train_fit, df_test):
    """Recursive evaluation on the true, never-touched-until-now holdout."""
    lag_positions = {i: features.index(f"{lag_prefix}_lag_{i}") for i in range(1, 13)
                     if f"{lag_prefix}_lag_{i}" in features}
    X_test = df_test[features].to_numpy(dtype=float)
    y_test = df_test[target].to_numpy(dtype=float)
    n = len(df_test)
    preds = np.empty(n)
    for month in range(n):
        row = X_test[month].copy()
        for lag_idx, col_pos in lag_positions.items():
            if month >= lag_idx:
                row[col_pos] = preds[month - lag_idx]
        preds[month] = model.predict(row.reshape(1, -1))[0]

    # the most recent few months may have no recorded flow yet (climate/
    # population data extends further than gage readings do) - score only
    # against months where we actually have ground truth.
    valid = ~np.isnan(y_test)
    holdout_rmse = root_mean_squared_error(y_test[valid], preds[valid])

    sa = SeasonalAverage(season_length=season_length)
    sa.fit(df_train_fit[target])
    sa_preds = sa.forecast(n)
    baseline_rmse = root_mean_squared_error(y_test[valid], sa_preds[valid])
    return holdout_rmse, baseline_rmse, preds


NICE_LABELS = {
    "log_flow_salt_upstream": "Upstream Flow",
    "log_flow_salt_downstream": "Downstream Flow",
    "Precip_Salt": "Watershed Precipitation",
    "Temp_Salt": "Watershed Temperature",
    "Temp_Maricopa": "Maricopa County Temperature",
    "Precip_Maricopa": "Maricopa County Precipitation",
    "Population": "Maricopa County Population",
    "IrrigatedLand_Acres": "Irrigated Farmland (Acres)",
    "datacenters_TotalMW": "Data Center Capacity (MW)",
    "datacenters_TotalNum": "Number of Data Centers",
}

def nice_label(col):
    """Turns a raw column name into a human-readable label for plots."""
    if col in NICE_LABELS:
        return NICE_LABELS[col]
    if "_lag_" in col:
        base, n = col.split("_lag_")
        n = int(n)
        direction = "Upstream" if "upstream" in base else "Downstream"
        unit = "Month" if n == 1 else "Months"
        return f"{direction} Flow, {n} {unit} Ago"
    return col.replace("_", " ")


def plot_importances(importances, title, save_name):
    fig, ax = plt.subplots(figsize=(7.5, 0.4 * len(importances) + 1))
    items = list(importances.items())[::-1]
    labels = [nice_label(k) for k, v in items]
    ax.barh(labels, [v for k, v in items], color="steelblue")
    ax.set_xlabel("Importance")
    ax.set_title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=300, bbox_inches="tight")
    plt.show()


def baseline_cv_rmse(target, data, tscv, season_length=12):
    """Seasonal-average baseline RMSE, evaluated across the same CV folds,
    for a fair side-by-side comparison in the output files, listed
    alongside every architecture's own result."""
    fold_rmses = []
    for train_idx, test_idx in tscv.split(data):
        train_series = data[target].iloc[train_idx]
        test_series = data[target].iloc[test_idx]
        sa = SeasonalAverage(season_length=season_length)
        sa.fit(train_series)
        preds = sa.forecast(len(test_idx))
        fold_rmses.append(root_mean_squared_error(test_series, preds))
    return np.mean(fold_rmses)

## Salt River Upstream Flow

Core features: `["Temp_Salt", "Precip_Salt"]` (watershed climate only - this is a natural hydrology process). Selecting most important lags.

In [ ]:
target = "log_flow_salt_upstream"
lag_prefix = "log_flow_salt_upstream"
core_features = ["Temp_Salt", "Precip_Salt"]
lag_pool = [f"{lag_prefix}_lag_{i}" for i in range(1, 13)]

models = {
    "randforest": RandomForestRegressor(n_jobs=-1, random_state=123),
    "xgboost": XGBRegressor(n_jobs=-1, random_state=42),
}

all_cols_needed = core_features + lag_pool + [target]
df_model_train = df_train.dropna(subset=all_cols_needed).reset_index(drop=True)
print(f"Modeling rows available: {len(df_model_train)}")

tscv = TimeSeriesSplit(n_splits=3)
n_lags_to_discover = 2

baseline_rmse_cv = baseline_cv_rmse(target, df_model_train, tscv)

selected_lags = {}
output_filename = os.path.join(OUTPUT_DIR, "salt_upstream_lag_selection.txt")
with open(output_filename, "w") as f:
    f.write(f"=== SALT RIVER UPSTREAM LAG SELECTION RESULTS ===\n")
    f.write(f"{'=' * 50}\n\n")

for name, model in models.items():
    print(f"Running lag selection for: {name}...")
    default_builder = lambda m=model: m.__class__(**m.get_params())
    chosen_lags, score = select_lags(default_builder, core_features, lag_pool, target, lag_prefix,
                                      df_model_train, tscv, n_lags=n_lags_to_discover)
    selected_lags[name] = chosen_lags
    with open(output_filename, "a") as f:
        f.write(f"Model: {name}\n")
        f.write(f"Discovered Optimal Lags: {chosen_lags}\n")
        f.write(f"Best Average RMSE: {score:.4f}\n")
        f.write(f"Benchmark Seasonal Average Baseline RMSE: {baseline_rmse_cv:.4f}\n")
        f.write(f"{'-' * 50}\n")
    print(f"  Selected lags: {chosen_lags}  (CV RMSE: {score:.4f})")

print(f"\nSaved to {output_filename}")

Modeling rows available: 1238
Running lag selection for: randforest...


The Salt upstream lag selection results can be found in `salt_upstream_lag_selection.txt`.

Next we do hyperparameter tuning for each model, with the best lags.

In [ ]:
grids = {
    "randforest": {"cls": RandomForestRegressor, "static": {"random_state": 123, "n_jobs": -1},
                    "grid": {"max_depth": [4, 6, 8, 10], "n_estimators": [100, 300]}},
    "xgboost": {"cls": XGBRegressor, "static": {"random_state": 42, "n_jobs": -1},
                 "grid": {"max_depth": [4, 6, 8, 10], "n_estimators": [100, 300]}},
}

best_per_arch = {}
output_filename = os.path.join(OUTPUT_DIR, "salt_upstream_grid_search_summary.txt")
with open(output_filename, "w") as f:
    f.write(f"=== SALT UPSTREAM GRID SEARCH RESULTS ===\n{'=' * 60}\n\n")

for name, info in grids.items():
    print(f"Running grid search for: {name}...")
    lags = selected_lags[name]
    best_params, best_score = grid_search(info["cls"], info["static"], info["grid"],
                                           core_features, lags, target, lag_prefix,
                                           df_model_train, tscv)
    best_per_arch[name] = {"params": best_params, "score": best_score, "lags": lags}
    with open(output_filename, "a") as f:
        f.write(f"Model Architecture: {name}\n")
        f.write(f"Best Parameters Found: {best_params}\n")
        f.write(f"Best Recursive Cross-Validation RMSE: {best_score:.4f}\n")
        f.write(f"Benchmark Seasonal Average Baseline RMSE: {baseline_rmse_cv:.4f}\n")
        f.write(f"{'-' * 70}\n")
    print(f"  Best params: {best_params}  (CV RMSE: {best_score:.4f})")

best_arch = min(best_per_arch, key=lambda k: best_per_arch[k]["score"])
print(f"\n>>> Best architecture for Salt upstream: {best_arch} (CV RMSE {best_per_arch[best_arch]['score']:.4f})")
print(f"Saved to {output_filename}")

The Salt upstream hyperparameter tuning results can be found in `salt_upstream_grid_search_summary.txt`.

Now we fit the final model on the training data, evaluate it on the held-out test set, and look at feature importance.

In [ ]:
final_lags = best_per_arch[best_arch]["lags"]
final_params = best_per_arch[best_arch]["params"]
final_features = core_features + final_lags

model_cls = grids[best_arch]["cls"]
static_params = grids[best_arch]["static"]
salt_upstream_model = model_cls(**{**static_params, **final_params})
salt_upstream_model.fit(df_model_train[final_features], df_model_train[target])

print(f"Final model: {best_arch}, params: {final_params}, lags: {final_lags}")

# ---- Evaluate on the test set ----
holdout_rmse, baseline_rmse, test_preds = evaluate_on_test_set(
    salt_upstream_model, final_features, target, lag_prefix, df_model_train, df_test)
print(f"\nHoldout RMSE (final model): {holdout_rmse:.4f}")
print(f"Holdout RMSE (seasonal average baseline): {baseline_rmse:.4f}")
print(f"Improvement over baseline: {(baseline_rmse - holdout_rmse) / baseline_rmse * 100:.1f}%")

# ---- Feature importance ----
importances = dict(sorted(
    {f: float(salt_upstream_model.feature_importances_[i]) for i, f in enumerate(final_features)}.items(),
    key=lambda item: item[1], reverse=True))
print("\nFeature importances:")
for f, v in importances.items():
    print(f"  {f:40s} {v:.4f}")
joblib.dump(salt_upstream_model, "salt_upstream_model.joblib")
plot_importances(importances, "Salt Upstream Flow - Feature Importance", "FeatureImportance_SaltUpstream.png")

Visualizes actual flow against the model's recursive predictions across the held-out test period, with a one-step-ahead seasonal baseline shown for comparison.

In [ ]:
# Test-set evaluation plot: actual flow vs. model predictions, with a
# one-step-ahead seasonal baseline (refit month by month) overlaid for
# comparison. Only genuinely valid (non-NaN) test months are plotted.
full_valid = df[df["log_flow_salt_upstream"].notna()][["Time", "log_flow_salt_upstream"]].reset_index(drop=True)
n_test_valid = df_test["log_flow_salt_upstream"].notna().sum()

baseline_preds_eval = []
for i in range(n_test_valid):
    sa_eval = SeasonalAverage(season_length=season_length)
    sa_eval.fit(full_valid["log_flow_salt_upstream"].iloc[:len(full_valid) - n_test_valid + i])
    baseline_preds_eval.append(sa_eval.forecast(1)[0])
baseline_preds_eval = np.array(baseline_preds_eval)

valid_mask = df_test["log_flow_salt_upstream"].notna().values
test_dates_valid = pd.to_datetime(df_test["Time"])[valid_mask].reset_index(drop=True)
actual_valid = df_test["log_flow_salt_upstream"][valid_mask].reset_index(drop=True)
preds_valid = test_preds[valid_mask]
baseline_rmse_eval = np.sqrt(np.mean((actual_valid.values - baseline_preds_eval) ** 2))

plt.figure(figsize=(14, 6))
plt.plot(test_dates_valid, actual_valid, label="Actual Flow (Ground Truth)", color="#2c3e50", linewidth=2.5, zorder=2)
plt.plot(test_dates_valid, preds_valid, label=f"{best_arch.title()} Model (RMSE: {holdout_rmse:.3f})", color="royalblue", linestyle="--", linewidth=2.5, zorder=3)
plt.plot(test_dates_valid, baseline_preds_eval, label=f"Seasonal Baseline (RMSE: {baseline_rmse_eval:.3f})", color="#95a5a6", linestyle=":", linewidth=1.5, zorder=1)

plt.title("Salt River Upstream Flow: Model Evaluated on Test Set", fontsize=14, fontweight="bold")
plt.xlabel("Timeline Horizon (Test Set)", fontsize=12)
plt.ylabel("Log Flow Volume", fontsize=12)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", fontsize=11, frameon=True, facecolor="white", edgecolor="none")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "TestSetEvaluation_SaltUpstream.png"), dpi=300, bbox_inches="tight")
plt.show()

Partial dependence: shows *how* each feature affects the prediction (direction and shape), which feature importance alone doesn't tell us.

In [ ]:
pdp_features = ["Temp_Salt", "Precip_Salt"]
pdp_labels = ["Watershed Temperature (\u00b0C)", "Watershed Precipitation (mm)"]

fig, ax = plt.subplots(figsize=(10, 3.5))
display = PartialDependenceDisplay.from_estimator(
    estimator=salt_upstream_model, X=df_model_train[final_features],
    features=pdp_features, ax=ax, grid_resolution=50)
for a, label in zip(display.axes_.ravel(), pdp_labels):
    a.set_title(label, fontsize=11)
    a.set_xlabel(label, fontsize=10)
    a.set_ylabel("Upstream Flow (log scale)", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "PartialDependency_SaltUpstream.png"), dpi=300, bbox_inches="tight")
plt.show()

## Salt River Downstream Flow

Core features: `["Temp_Salt", "Precip_Salt", "Temp_Maricopa", "Precip_Maricopa", "Population", "IrrigatedLand_Acres", "datacenters_TotalMW", "datacenters_TotalNum", "log_flow_salt_upstream"]` (current-month upstream flow is included directly (not lagged). Selecting most important lags.

In [ ]:
target = "log_flow_salt_downstream"
lag_prefix = "log_flow_salt_downstream"
core_features = ["Temp_Salt", "Precip_Salt", "Temp_Maricopa", "Precip_Maricopa", "Population", "IrrigatedLand_Acres", "datacenters_TotalMW", "datacenters_TotalNum", "log_flow_salt_upstream"]
lag_pool = [f"{lag_prefix}_lag_{i}" for i in range(1, 13)]

models = {
    "randforest": RandomForestRegressor(n_jobs=-1, random_state=123),
    "xgboost": XGBRegressor(n_jobs=-1, random_state=42),
}

all_cols_needed = core_features + lag_pool + [target]
df_model_train = df_train.dropna(subset=all_cols_needed).reset_index(drop=True)
print(f"Modeling rows available: {len(df_model_train)}")

tscv = TimeSeriesSplit(n_splits=3)
n_lags_to_discover = 2

baseline_rmse_cv = baseline_cv_rmse(target, df_model_train, tscv)

selected_lags = {}
output_filename = os.path.join(OUTPUT_DIR, "salt_downstream_lag_selection.txt")
with open(output_filename, "w") as f:
    f.write(f"=== SALT RIVER DOWNSTREAM LAG SELECTION RESULTS ===\n")
    f.write(f"{'=' * 50}\n\n")

for name, model in models.items():
    print(f"Running lag selection for: {name}...")
    default_builder = lambda m=model: m.__class__(**m.get_params())
    chosen_lags, score = select_lags(default_builder, core_features, lag_pool, target, lag_prefix,
                                      df_model_train, tscv, n_lags=n_lags_to_discover)
    selected_lags[name] = chosen_lags
    with open(output_filename, "a") as f:
        f.write(f"Model: {name}\n")
        f.write(f"Discovered Optimal Lags: {chosen_lags}\n")
        f.write(f"Best Average RMSE: {score:.4f}\n")
        f.write(f"Benchmark Seasonal Average Baseline RMSE: {baseline_rmse_cv:.4f}\n")
        f.write(f"{'-' * 50}\n")
    print(f"  Selected lags: {chosen_lags}  (CV RMSE: {score:.4f})")

print(f"\nSaved to {output_filename}")

The Salt downstream lag selection results can be found in `salt_downstream_lag_selection.txt`.

Next we do hyperparameter tuning for each model, with the best lags.

In [ ]:
grids = {
    "randforest": {"cls": RandomForestRegressor, "static": {"random_state": 123, "n_jobs": -1},
                    "grid": {"max_depth": [4, 6, 8, 10], "n_estimators": [100, 300]}},
    "xgboost": {"cls": XGBRegressor, "static": {"random_state": 42, "n_jobs": -1},
                 "grid": {"max_depth": [4, 6, 8, 10], "n_estimators": [100, 300]}},
}

best_per_arch = {}
output_filename = os.path.join(OUTPUT_DIR, "salt_downstream_grid_search_summary.txt")
with open(output_filename, "w") as f:
    f.write(f"=== SALT DOWNSTREAM GRID SEARCH RESULTS ===\n{'=' * 60}\n\n")

for name, info in grids.items():
    print(f"Running grid search for: {name}...")
    lags = selected_lags[name]
    best_params, best_score = grid_search(info["cls"], info["static"], info["grid"],
                                           core_features, lags, target, lag_prefix,
                                           df_model_train, tscv)
    best_per_arch[name] = {"params": best_params, "score": best_score, "lags": lags}
    with open(output_filename, "a") as f:
        f.write(f"Model Architecture: {name}\n")
        f.write(f"Best Parameters Found: {best_params}\n")
        f.write(f"Best Recursive Cross-Validation RMSE: {best_score:.4f}\n")
        f.write(f"Benchmark Seasonal Average Baseline RMSE: {baseline_rmse_cv:.4f}\n")
        f.write(f"{'-' * 70}\n")
    print(f"  Best params: {best_params}  (CV RMSE: {best_score:.4f})")

best_arch = min(best_per_arch, key=lambda k: best_per_arch[k]["score"])
print(f"\n>>> Best architecture for Salt downstream: {best_arch} (CV RMSE {best_per_arch[best_arch]['score']:.4f})")
print(f"Saved to {output_filename}")

The Salt downstream hyperparameter tuning results can be found in `salt_downstream_grid_search_summary.txt`.

Now we fit the final model on the training data, evaluate it on the held-out test set, and look at feature importance.

In [ ]:
final_lags = best_per_arch[best_arch]["lags"]
final_params = best_per_arch[best_arch]["params"]
final_features = core_features + final_lags

model_cls = grids[best_arch]["cls"]
static_params = grids[best_arch]["static"]
salt_downstream_model = model_cls(**{**static_params, **final_params})
salt_downstream_model.fit(df_model_train[final_features], df_model_train[target])

print(f"Final model: {best_arch}, params: {final_params}, lags: {final_lags}")

# ---- Evaluate on the test set ----
holdout_rmse, baseline_rmse, test_preds = evaluate_on_test_set(
    salt_downstream_model, final_features, target, lag_prefix, df_model_train, df_test)
print(f"\nHoldout RMSE (final model): {holdout_rmse:.4f}")
print(f"Holdout RMSE (seasonal average baseline): {baseline_rmse:.4f}")
print(f"Improvement over baseline: {(baseline_rmse - holdout_rmse) / baseline_rmse * 100:.1f}%")

# ---- Feature importance ----
importances = dict(sorted(
    {f: float(salt_downstream_model.feature_importances_[i]) for i, f in enumerate(final_features)}.items(),
    key=lambda item: item[1], reverse=True))
print("\nFeature importances:")
for f, v in importances.items():
    print(f"  {f:40s} {v:.4f}")
joblib.dump(salt_downstream_model, "salt_downstream_model.joblib")
plot_importances(importances, "Salt Downstream Flow - Feature Importance", "FeatureImportance_SaltDownstream.png")

Same evaluation as above, for the downstream model.

In [ ]:
# Test-set evaluation plot: actual flow vs. model predictions, with a
# one-step-ahead seasonal baseline (refit month by month) overlaid for
# comparison. Only genuinely valid (non-NaN) test months are plotted.
full_valid = df[df["log_flow_salt_downstream"].notna()][["Time", "log_flow_salt_downstream"]].reset_index(drop=True)
n_test_valid = df_test["log_flow_salt_downstream"].notna().sum()

baseline_preds_eval = []
for i in range(n_test_valid):
    sa_eval = SeasonalAverage(season_length=season_length)
    sa_eval.fit(full_valid["log_flow_salt_downstream"].iloc[:len(full_valid) - n_test_valid + i])
    baseline_preds_eval.append(sa_eval.forecast(1)[0])
baseline_preds_eval = np.array(baseline_preds_eval)

valid_mask = df_test["log_flow_salt_downstream"].notna().values
test_dates_valid = pd.to_datetime(df_test["Time"])[valid_mask].reset_index(drop=True)
actual_valid = df_test["log_flow_salt_downstream"][valid_mask].reset_index(drop=True)
preds_valid = test_preds[valid_mask]
baseline_rmse_eval = np.sqrt(np.mean((actual_valid.values - baseline_preds_eval) ** 2))

plt.figure(figsize=(14, 6))
plt.plot(test_dates_valid, actual_valid, label="Actual Flow (Ground Truth)", color="#2c3e50", linewidth=2.5, zorder=2)
plt.plot(test_dates_valid, preds_valid, label=f"{best_arch.title()} Model (RMSE: {holdout_rmse:.3f})", color="royalblue", linestyle="--", linewidth=2.5, zorder=3)
plt.plot(test_dates_valid, baseline_preds_eval, label=f"Seasonal Baseline (RMSE: {baseline_rmse_eval:.3f})", color="#95a5a6", linestyle=":", linewidth=1.5, zorder=1)

plt.title("Salt River Downstream Flow: Model Evaluated on Test Set", fontsize=14, fontweight="bold")
plt.xlabel("Timeline Horizon (Test Set)", fontsize=12)
plt.ylabel("Log Flow Volume", fontsize=12)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", fontsize=11, frameon=True, facecolor="white", edgecolor="none")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "TestSetEvaluation_SaltDownstream.png"), dpi=300, bbox_inches="tight")
plt.show()

Partial dependence for downstream - the substantive demand-driver features (climate, population, irrigation, data centers), leaving out the lag features since their effect is mostly mechanical persistence rather than something interesting to visualize.

In [ ]:
pdp_features = ["log_flow_salt_upstream", "Population", "IrrigatedLand_Acres",
                 "Temp_Maricopa", "Precip_Maricopa", "datacenters_TotalMW"]
pdp_labels = ["Upstream Flow (log scale)", "Maricopa County Population", "Irrigated Farmland (Acres)",
              "Maricopa County Temperature (\u00b0C)", "Maricopa County Precipitation (mm)",
              "Data Center Capacity (MW)"]

fig, ax = plt.subplots(2, 3, figsize=(14, 7))
display = PartialDependenceDisplay.from_estimator(
    estimator=salt_downstream_model, X=df_model_train[final_features],
    features=pdp_features, ax=ax, grid_resolution=50)
for a, label in zip(display.axes_.ravel(), pdp_labels):
    a.set_title(label, fontsize=11)
    a.set_xlabel(label, fontsize=9)
    a.set_ylabel("Downstream Flow (log scale)", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "PartialDependency_SaltDownstream.png"), dpi=300, bbox_inches="tight")
plt.show()